# 📊 Measuring How Well a Model Fits: RMSE
### EPS Research High-School Exploration Track — Ages 15-18

In science, we need a way to measure how well a model fits data.
The most common metric is **RMSE** (Root Mean Square Error):

$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(V_{\rm model,i} - V_{\rm data,i})^2}$$

Lower RMSE = better fit. Let's compute it for DDO161.

**Prerequisites:** Basic statistics, square roots, summation notation

In [1]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21327061/files/high_z_kinematic_corpus_Z1.json',
        'dwarf_irregular_corpus_v1.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


Running locally — corpus files loaded from working directory.


In [2]:
import json, numpy as np, matplotlib.pyplot as plt
with open('rotation_curve_corpus_v7.json') as f:
    corpus=json.load(f)
g=next(g for g in corpus['galaxies'] if g['galaxy']=='DDO161')
d=g['data']
R=np.array([p['Rad'] for p in d]); V=np.array([p['Vobs'] for p in d])
Vgas=np.array([p['Vgas'] for p in d]); Vdisk=np.array([p['Vdisk'] for p in d])
Vbul=np.array([p['Vbul'] for p in d])
Vbar=np.where(Vgas<0,-np.sqrt(Vgas**2+Vdisk**2+Vbul**2),np.sqrt(Vgas**2+Vdisk**2+Vbul**2))
R1,V1=R[0],V[0]; R2,V2=R[-1],V[-1]
omega=V2/R2 - (V1/R1)*(R1/R2)**1.5  # Eq.6 corrected 2026-07-12: operator-precedence fix
V_adj=V-R*omega
V_kep=np.sqrt(V2**2*R2/R)

def rmse(pred, true):
    return np.sqrt(np.mean((pred - true)**2))

print("RMSE comparison for DDO161:")
print(f"  Keplerian model:   RMSE = {rmse(V_kep, Vbar):.2f} km/s  (pure Newtonian, no dark matter)")
print(f"  Omega correction:  RMSE = {rmse(V_adj, Vbar):.2f} km/s  (EPS Research correction)")
print(f"  Improvement:       {rmse(V_kep, Vbar) - rmse(V_adj, Vbar):.2f} km/s better")
print()
print("For the full 84-galaxy SPARC sample (Flynn 2026):")
print("  Mean RMSE (Keplerian): 74.20 km/s")
print("  Mean RMSE (omega):     25.45 km/s")
print("  That's a 2× average improvement!")

RMSE comparison for DDO161:
  Keplerian model:   RMSE = 121.27 km/s  (pure Newtonian, no dark matter)
  Omega correction:  RMSE = 14.55 km/s  (EPS Research correction)
  Improvement:       106.72 km/s better

For the full 84-galaxy SPARC sample (Flynn 2026):
  Mean RMSE (Keplerian): 74.20 km/s
  Mean RMSE (omega):     25.45 km/s
  That's a 2× average improvement!
